## 1)LIBRARY AND ACCELERATOR CHOICE

---

We import the necessary libraries and the model architecture we want to quantize (MNIST CNN or CIFAR10 CNN). We load the weights trained earlier in the workflow. 
To compare the accuracy change when we go from FP32 accuracy to INT8/4 accuracy, we make a deepcopy of the CNN where the weights are quantized.

All computations run on the CPU because there is no training so there is no need for an heavy computationnal power.

In [27]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import sys
sys.path.append("../src")
from model import CNN_MNIST  #CNN_CIFAR10 for the CIFAR10 CNN
from copy import deepcopy

device="cpu"

In [37]:
model = CNN_MNIST().to(device)
state_dict = torch.load("../../checkpoints/mnist_fp32.pt", map_location=device)
model.load_state_dict(state_dict)
model.eval()

CNN_MNIST(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

## 2)DATA

---

We need to normalise the data because the model is trained on normalized data. We also need to choose whether we want to have INT4 or INT8 quantization.

In [ ]:


quantization = 4       # 4 or 8
assert quantization in (4, 8)

DATASET = "MNIST"   # "MNIST" or "CIFAR10"
assert DATASET in ("MNIST", "CIFAR10")


qmax = 2**(quantization - 1) - 1       # 7 in INT4, 127 in INT8




if DATASET == "MNIST":
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_data = datasets.MNIST(root="data", train=True,  download=True, transform=transform)
    test_data  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader  = DataLoader(test_data,  batch_size=64)
    
elif DATASET == "CIFAR10":
    Mean = torch.tensor([0.4914, 0.4822, 0.4465]) 
    Dev  = torch.tensor([0.2470, 0.2435, 0.2616])

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4), #the RandomCrop and HorizontalFlip are standards on CNN for the CIFAR-10 dataset
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(Mean, Dev),
    ])


    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(Mean, Dev),
    ])

    train_data = datasets.CIFAR10(root="data", train=True,  download=True, transform=train_tf)
    test_data  = datasets.CIFAR10(root="data", train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader  = DataLoader(test_data,  batch_size=64)


## 3)WEIGHTS QUANTIZATION

---
The weight are quantized by layer. We use a scale stocked in a dictionnary because the calculation is still made in FP32 and need to fit this format. 

In [61]:
model_q = deepcopy(model)
q_scales = {}

with torch.no_grad():
    for name, m in model_q.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            w = m.weight
            scale = w.abs().max() / qmax
            q = torch.round(w / scale).clamp(-qmax, qmax)
            w.copy_(q * scale)
            q_scales[name] = scale

## 4)INPUTS QUANTIZATION

---
Now that the weights are quantize, we need to quantize the inputs (activations) of each layer as the input in the systolic array are also in INT8 or INT4. Therefore we need to measure, with a few batch, the range of the activations of each layer, so the quantization is well-sized.

To do this, we first use a hook that will take the maximum and minimum of the activations of each layer and store it in a dictionnary.
After that we create a new hook that ensure to quantize the activation of each layer using the dictionnary.

In [62]:
act_range = {}

def make_range_hook(name):
    def hook(module, input):
        x = input[0]
        old_min, old_max = act_range.get(name, (float("inf"), float("-inf")))
        act_range[name] = (min(old_min, x.min().item()),
                           max(old_max, x.max().item()))
    return hook

handlers = []
for name, m in model_q.named_modules():
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        handlers.append(m.register_forward_pre_hook(make_range_hook(name)))

with torch.no_grad():
    for i, (x, _) in enumerate(train_loader):
        model_q(x.to(device))
        if i == 50:
            break

for h in handlers:
    h.remove()

In [63]:
def make_quant_hook(name):
    lo, hi = act_range[name]

    def hook(module, input):
        x = input[0]
        q = torch.round((x - lo) / (hi - lo) * 2 * qmax - qmax).clamp(-qmax, qmax)
        return (q + qmax) / (2 * qmax) * (hi - lo) + lo
    return hook

quant_handles = []
for name, m in model_q.named_modules():
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        quant_handles.append(m.register_forward_pre_hook(make_quant_hook(name)))

## 5) COMPARISON

---

We want to measure the total accuracy and the error induced by layer. So for the total accuracy we just count the number of times the model is false on the test data. And the error is measured layer by layer with a `forward_hook` at the end of every layer for the quantized model and the original model. As the operation of quantization is akin to a CAN, we use the signal-to-quantization-noise ratio (`SQNR`) to help identify the most degraded layer.


In [64]:
def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


def capture(model, x):
    outputs = {}
    def make_hook(name):
        def hook(module, input, output):
            outputs[name] = output.clone()
        return hook
    handles = [m.register_forward_hook(make_hook(n))
               for n, m in model.named_modules()
               if isinstance(m, (nn.Conv2d, nn.Linear))]
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()         
    return outputs

In [65]:
x, _ = next(iter(test_loader))
x = x.to(device)

out_fp = capture(model, x)
out_q  = capture(model_q, x)

for name in out_fp:
    ref, q = out_fp[name], out_q[name]
    err_rel = ((ref - q).abs().mean() / ref.abs().mean()).item()
    sqnr = 10 * torch.log10(ref.pow(2).mean() / (ref - q).pow(2).mean()).item()
    print(f"{name:10s} erreur relative = {err_rel:6.2%}   SQNR = {sqnr:5.1f} dB")

conv1      erreur relative =  9.50%   SQNR =  20.4 dB
conv2      erreur relative = 10.30%   SQNR =  20.0 dB
fc1        erreur relative = 22.26%   SQNR =  12.7 dB
fc2        erreur relative = 10.57%   SQNR =  19.7 dB


In [66]:
print(f"Accuracy FP32: {accuracy(model, test_loader):.4f}")
print(f"Accuracy INT{quantization}: {accuracy(model_q, test_loader):.4f}") 

Accuracy FP32: 0.9908
Accuracy INT4: 0.9886


## 6)RESULTS
---


In [67]:
diff, total = 0, 0
with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(device)
        diff += (model(x).argmax(1) != model_q(x).argmax(1)).sum().item()
        total += x.size(0)
print(f"Different predictions : {diff} / {total}")

Different predictions : 44 / 10000
